# 04 — Low-Memory Dropout

**Ladder:** Naive materialize mask → fused rand+scale (Python) → **seeded Philox** → Triton `tl.rand` (no mask tensor).

**Study tip.** Prove $\mathbb{E}[Y]=X$ for inverted dropout, then ask what you store instead of an $O(N)$ mask (the seed). Forward/backward must share that seed.

Official: [Low-Memory Dropout](https://triton-lang.org/main/getting-started/tutorials/04-low-memory-dropout.html)

Figures: [`docs/images/`](docs/images/) · guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

## 0. Definition (inverted dropout)

Fix keep-probability $q = 1-p$ with $p \in [0,1)$. Let $M_i \sim \mathrm{Bernoulli}(q)$ i.i.d. Define

$$
Y_i
=
\begin{cases}
X_i / q & \text{if } M_i = 1, \\
0 & \text{if } M_i = 0.
\end{cases}
$$

Equivalently $Y_i = \dfrac{M_i}{1-p}\, X_i$. At eval, $p=0$ and $Y=X$.

**Proposition (unbiasedness).** $\mathbb{E}[Y_i \mid X_i] = X_i$.

*Proof.*

$$
\mathbb{E}[Y_i \mid X_i]
=
q \cdot \frac{X_i}{q} + (1-q)\cdot 0
= X_i.
$$

**Memory.** Storing $M \in \{0,1\}^{N}$ costs $\Theta(N)$ extra bytes and an extra full read in the apply pass. Low-memory dropout never materializes $M$: it samples $U_i \sim \mathrm{Unif}(0,1)$ on the fly and sets $M_i = \mathbf{1}[U_i > p]$.

![Whiteboard: materialize mask vs Philox seed in registers](docs/images/benchmark-low-memory-dropout.png)

*Theory board — $O(N)$ mask tensor in DRAM vs seed + `Philox(s,i)` keep/drop in registers.*

### Theory board

![Whiteboard: materialize mask vs Philox seed in registers](docs/images/whiteboard-theory.jpg)

*Theory board — $O(N)$ mask tensor in DRAM vs seed + `Philox(s,i)` keep/drop in registers.*

### Pitfalls

- Using p vs 1-p incorrectly in the keep test.
- Materializing a full mask defeats the point of this tutorial.
- Different seeds → different masks; compare distributions, not exact tensors.

### Self-check

1. Show E[Y]=X under inverted dropout.
2. Where does the boolean keep decision live in the Triton kernel?
3. Why is a single int seed enough to regenerate the mask?


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



## 1. Naive — materialize $M$, then scale

$$
U_i \sim \mathrm{Unif}(0,1),\qquad
M_i = \mathbf{1}[U_i > p],\qquad
Y = \frac{X \odot M}{1-p}.
$$

**Teaching sizes.** Try $N=8$ or $N=1024$, $p=0.5$. Keep-rate should track $1-p$; mean of many seeded runs of $Y$ should track $X$.

Correct in law, but writes/reads a full mask tensor — roughly doubles traffic relative to a fused keep/drop that never stores $M$.


In [ ]:
def naive_dropout_pseudo(x, p, seed: int = 0):
    """
    Classic dropout: allocate a full keep-mask, then apply it.

    Memory picture for an n-element tensor:
        - read  x          (n floats)
        - write mask       (n bools)   <-- extra global buffer
        - read  x + mask
        - write y

    The mask itself is often as large as x. That is the cost we want to avoid.
    """
    assert 0.0 <= p < 1.0
    g = torch.Generator(device="cpu")
    g.manual_seed(seed)
    # Materialize the full Bernoulli mask in memory
    x_cpu = x.detach().cpu()
    mask = torch.rand(x_cpu.shape, generator=g, dtype=x_cpu.dtype) > p
    y = torch.where(mask, x_cpu / (1.0 - p), torch.zeros_like(x_cpu))
    return y.to(device=x.device, dtype=x.dtype)


## 2–3. Fused form → Triton Philox

### Intermediate (Python fused)

Per index $i$, one pass — still host RNG, but no persistent mask array (`fused_dropout_pseudo`):

$$
U_i = \mathrm{rand}(\mathrm{seed}, i),\qquad
Y_i = \mathbf{1}[U_i > p]\,\frac{X_i}{1-p}.
$$

### Triton (seeded, in-kernel)

Philox / hash RNG keyed by $(\mathrm{seed}, i)$ via `tl.rand(seed, offsets)`:

$$
U_i = \mathrm{tl.rand}(\mathrm{seed}, i),\qquad
Y_i = \mathbf{1}[U_i > p]\,\frac{X_i}{1-p}.
$$

Same distribution as §1 if $U_i$ are i.i.d. $\mathrm{Unif}(0,1)$. Uniforms stay in **registers** — **no** global $M$.

**Reproducibility.** Same seed $\Rightarrow$ same keep-set on forward, backward, or activation-checkpoint recompute. Only $O(1)$ RNG state is stored.

| Stage | Function | Mask in DRAM? |
|-------|----------|---------------|
| Naive | `naive_dropout_pseudo` | yes |
| Fused Python | `fused_dropout_pseudo` | no (transient) |
| Triton | `dropout_kernel` / `dropout` | no |


In [ ]:
def fused_dropout_pseudo(x, p, seed: int = 0):
    """
    Same math, but generate the keep decision per element without storing mask.

    We use a Python loop (for teaching): random() and the scale/zero happen in
    one pass — no n-element mask tensor is ever allocated. The Triton kernel
    does the same with tl.rand(seed, offsets).
    """
    assert 0.0 <= p < 1.0
    n = x.numel()
    y = torch.empty_like(x, device="cpu").cpu()
    x_flat = x.detach().cpu().reshape(-1)
    y_flat = y.reshape(-1)

    g = torch.Generator(device="cpu")
    g.manual_seed(seed)

    for i in range(n):
        # Uniform[0, 1) keep decision — never written to a mask buffer
        r = torch.rand((), generator=g).item()
        if r > p:
            y_flat[i] = x_flat[i] / (1.0 - p)
        else:
            y_flat[i] = 0.0

    return y.to(device=x.device, dtype=x.dtype)


In [ ]:
@triton.jit
def dropout_kernel(
    x_ptr,
    output_ptr,
    n_elements,
    p,
    seed,
    BLOCK_SIZE: tl.constexpr,
):
    # Blocked 1-D grid (same pattern as vector-add)
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    # On-the-fly RNG — no mask tensor in HBM
    random = tl.rand(seed, offsets)
    x_keep = random > p
    output = tl.where(x_keep, x / (1 - p), 0.0)
    tl.store(output_ptr + offsets, output, mask=mask)


def dropout(x, p, seed=0):
    output = torch.empty_like(x)
    n_elements = x.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
    dropout_kernel[grid](x, output, n_elements, p, seed, BLOCK_SIZE=1024)
    return output


In [ ]:
def test_pseudo_statistical() -> None:
    """CPU-runnable: keep rate ≈ (1-p); mean ≈ 1 on a ones vector; seeded fuse is reproducible."""
    torch.manual_seed(0)
    n = 50_000
    x = torch.ones(n)  # ones → expected mean is 1 under inverted dropout
    p = 0.25

    y_naive = naive_dropout_pseudo(x, p, seed=0)
    y_fused = fused_dropout_pseudo(x, p, seed=7)

    keep_naive = (y_naive != 0).float().mean().item()
    keep_fused = (y_fused != 0).float().mean().item()
    assert abs(keep_naive - (1.0 - p)) < 0.02
    assert abs(keep_fused - (1.0 - p)) < 0.02

    # Inverted dropout on ones → kept values are 1/(1-p); overall mean ≈ 1
    assert abs(y_naive.mean().item() - 1.0) < 0.05
    assert abs(y_fused.mean().item() - 1.0) < 0.05

    # p=0 → identity
    assert torch.allclose(naive_dropout_pseudo(x, p=0.0, seed=0), x)
    assert torch.allclose(fused_dropout_pseudo(x, p=0.0, seed=0), x)

    # Seeded reproducibility for fused Python path
    y_fused_2 = fused_dropout_pseudo(x, p, seed=7)
    assert torch.equal(y_fused, y_fused_2)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(4096, device=DEVICE)
    p = 0.25
    seed = 123
    y = dropout(x, p, seed=seed)
    y2 = dropout(x, p, seed=seed)
    # Same seed → bitwise-identical output (tl.rand is deterministic given seed+offset)
    assert torch.equal(y, y2)

    # Kept values must be x / (1-p)
    scale = 1.0 / (1.0 - p)
    nonzero = y != 0
    assert torch.allclose(y[nonzero], x[nonzero] * scale, atol=1e-5)

    # Statistical sanity on a larger buffer of ones
    ones = torch.ones(100_000, device=DEVICE)
    y_stat = dropout(ones, p, seed=seed)
    keep_rate = (y_stat != 0).float().mean().item()
    assert abs(keep_rate - (1.0 - p)) < 0.02
    assert abs(y_stat.mean().item() - 1.0) < 0.05
    print("deterministic dropout OK", y.count_nonzero().item())
